In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
df_fixture = pd.read_csv("fixtures.csv")
df_fixture.head()

In [ ]:
df_team = pd.read_csv("teams.csv")
df_team.head()

In [ ]:
name_map = df_team.set_index("id")["name"]
short_map = df_team.set_index("id")["short_name"]

df_fixture["home_name"] = df_fixture["home_id"].map(name_map)
df_fixture["home_short_name"] = df_fixture["home_id"].map(short_map)
df_fixture["away_name"] = df_fixture["away_id"].map(name_map)
df_fixture["away_short_name"] = df_fixture["away_id"].map(short_map)

In [ ]:
def prediction_accurate(row) -> bool:
    if row["home_score"] > row["away_score"]:
        return row["home_difficulty"] < row["away_difficulty"]
    if row["home_score"] < row["away_score"]:
        return row["home_difficulty"] > row["away_difficulty"]
    if row["home_score"] == row["away_score"]:
        return row["home_difficulty"] == row["away_difficulty"]
    return False


df_fixture["prediction_accurate"] = df_fixture.apply(prediction_accurate, axis=1)

In [ ]:
df_fixture["away_cs"] = df_fixture["home_score"] == 0
df_fixture["home_cs"] = df_fixture["away_score"] == 0

df_fixture["home_gd"] = df_fixture["home_score"] - df_fixture["away_score"]
df_fixture["away_gd"] = df_fixture["away_score"] - df_fixture["home_score"]

In [ ]:
accuracy_by_gw = df_fixture.groupby("Gameweek")["prediction_accurate"].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=accuracy_by_gw, x="Gameweek", y="prediction_accurate", marker="o", ax=ax)

ax.set_title("Prediction Accuracy by Gameweek")
ax.set_xlabel("Gameweek")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
overall_accuracy = df_fixture["prediction_accurate"].mean()
ax.axhline(overall_accuracy, color="red", linestyle="--", linewidth=2, label=f"Season Avg ({overall_accuracy:.1%})")
ax.grid(True, alpha=0.7)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
game_counts = pd.crosstab(df_fixture["away_difficulty"], df_fixture["home_difficulty"])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(game_counts, annot=True, fmt="d", cmap="coolwarm", ax=ax)

ax.set_title("Home and Away Difficulty Occurences")
ax.set_xlabel("Home Difficulty")
ax.set_ylabel("Away Difficulty")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
home_goals = pd.pivot_table(
    df_fixture,
    values="home_score",
    index="away_difficulty",
    columns="home_difficulty",
    aggfunc="mean",
    fill_value=0,
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(home_goals, annot=True, fmt=".2f", cmap="coolwarm", ax=ax)

ax.set_title("Home Goals per Game")
ax.set_xlabel("Home Difficulty")
ax.set_ylabel("Away Difficulty")
ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
away_goals = pd.pivot_table(
    df_fixture,
    values="away_score",
    index="away_difficulty",
    columns="home_difficulty",
    aggfunc="mean",
    fill_value=0,
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(away_goals, annot=True, fmt=".2f", cmap="coolwarm", ax=ax)

ax.set_title("Away Goals per Game")
ax.set_xlabel("Home Difficulty")
ax.set_ylabel("Away Difficulty")
ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
home_cs = pd.pivot_table(
    df_fixture,
    values="home_cs",
    index="away_difficulty",
    columns="home_difficulty",
    aggfunc="mean",
    fill_value=0,
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(home_cs, annot=True, fmt=".2f", cmap="coolwarm", ax=ax)

ax.set_title("Home Clean Sheets %")
ax.set_xlabel("Home Difficulty")
ax.set_ylabel("Away Difficulty")
ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
away_cs = pd.pivot_table(
    df_fixture,
    values="away_cs",
    index="away_difficulty",
    columns="home_difficulty",
    aggfunc="mean",
    fill_value=0,
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(away_cs, annot=True, fmt=".2f", cmap="coolwarm", ax=ax)

ax.set_title("Away Clean Sheets %")
ax.set_xlabel("Home Difficulty")
ax.set_ylabel("Away Difficulty")
ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
df_fixture[df_fixture["home_difficulty"] > df_fixture["away_difficulty"]]["away_cs"].mean()

In [ ]:
df_fixture[(df_fixture["home_difficulty"] - df_fixture["away_difficulty"]) <= -2]["home_cs"].mean()